<a href="https://colab.research.google.com/github/internetOnion/machine-learning-101/blob/main/cat_or_dog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Is the image a dog or a cat or something else? using image binary classification model

## Setting up environments

In [ ]:
!python --version
!pip --version

In [ ]:
!pip install -U fastai "fastcore<2.0" ddgs kagglehub

## Download datasets

In [ ]:
import kagglehub

cat_and_dog_path = kagglehub.dataset_download("tongpython/cat-and-dog")

print(f"Path to dataset files: {cat_and_dog_path}")

In [ ]:
from fastcore.all import *
from fastai.vision.all import *
from fastdownload import download_url

In [ ]:
dest_cat = Path('cat')
dest_cat.mkdir(exist_ok=True, parents=True)
dest_dog = Path('dog')
dest_dog.mkdir(exist_ok=True, parents=True)

resize_images(
  Path(cat_and_dog_path, 'test_set/test_set/cats'),
  max_size=400,
  dest=Path(dest_cat)
)
resize_images(
  Path(cat_and_dog_path, 'training_set/training_set/cats'),
  max_size=400,
  dest=Path(dest_cat)
)

resize_images(
  Path(cat_and_dog_path, 'test_set/test_set/dogs'),
  max_size=400,
  dest=Path(dest_dog)
)
resize_images(
  Path(cat_and_dog_path, 'training_set/training_set/dogs'),
  max_size=400,
  dest=Path(dest_dog)
)

## Train model

In [ ]:
dls = DataBlock(
  blocks=(ImageBlock, CategoryBlock),
  get_items=get_image_files,
  splitter=RandomSplitter(valid_pct=0.2, seed=42),
  get_y=parent_label,
  item_tfms=[Resize(192, method='squish')],
  batch_tfms=aug_transforms()
).dataloaders(Path('.'), bs=128)

dls.show_batch(max_n=9)

In [ ]:
print(f"train: {len(dls.train_ds)}")
print(f"valid: {len(dls.valid_ds)}")

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(7)

In [ ]:
learn.recorder.plot_loss()

## Using model

In [ ]:
model_export_name = 'cat_or_dog.pkl'
learn.export(model_export_name)

learn_inf = load_learner(model_export_name)

print(f"Model loaded successfully from '{model_export_name}'")

In [ ]:
image_to_predict_path = 'to-path'
prediction, _, probabilities = learn_inf.predict(image_to_predict_path)

print(f"Image: {image_to_predict_path}")
print(f"Predicted class: {prediction}")
print(f"Probabilities: {probabilities}")

img = PILImage.create('test-cat.jpeg')
img.show(title=f'Prediction: {prediction} (Prob: {probabilities.max():.2f})')